# A RAG API skeleton, end to end

This notebook runs the FastAPI programs of the lesson *Build a RAG API skeleton end to end*. Run the cells in order. No GPU and no API key are needed (the keys in the code are invented demo values), and no server is started: `TestClient` calls the application inside the notebook. The retrieval and the model are a stand-in that returns fixed sentences. The outputs saved in the cells come from a real run with the pinned versions.

In [ ]:
!pip install -q fastapi==0.141.1 httpx==0.28.1 pydantic==2.14.3
print('installed')

## 1. What you get by only declaring types

In [ ]:
from fastapi import Depends, FastAPI, Header, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

class FirstRequest(BaseModel):
    question: str
    top_k: int = 4

first = FastAPI()

def require_key(authorization: str = Header(default="")):
    if authorization != "Bearer demo-key-acme":            # invented demo key
        raise HTTPException(401, "unknown key")

@first.post("/v1/ask", dependencies=[Depends(require_key)])
def first_ask(body: FirstRequest):
    return {"top_k_used": body.top_k}

client = TestClient(first)                                  # calls the app in this process: no server, no port
KEY = {"Authorization": "Bearer demo-key-acme", "Content-Type": "application/json"}

def show(label, content, headers=KEY):
    r = client.post("/v1/ask", content=content, headers=headers)
    body = r.json()
    if isinstance(body.get("detail"), list):                # shorten FastAPI's list to type and place
        body = {"detail": [(d["type"], d["loc"][-1]) for d in body["detail"]]}
    print(f"{label:20} {r.status_code} {body}")

show("no question", '{}')
show("top_k is true", '{"question": "leave?", "top_k": true}')
show('top_k is "7"', '{"question": "leave?", "top_k": "7"}')
show("tenant in the body", '{"question": "leave?", "tenant": "zen"}')
show("broken JSON, no key", '{oops', {"Content-Type": "application/json"})

no question          422 {'detail': [('missing', 'question')]}
top_k is true        200 {'top_k_used': 1}
top_k is "7"         200 {'top_k_used': 7}
tenant in the body   200 {'top_k_used': 4}
broken JSON, no key  422 {'detail': [('json_invalid', 1)]}


## 2. The whole service: stand-in pipeline, contract, strict schema and application

In [ ]:
import io
import json
import re

# STAND-IN for retrieve + prompt + model: it looks for a phrase and returns a fixed sentence with the document it came from
FACTS = [("annual leave", "Full-time employees receive 24 days of paid annual leave per calendar year.", "leave-policy"),
         ("sick", "Up to 10 paid sick days per calendar year need no medical certificate.", "leave-policy"),
         ("deleted ticket", "Deleted tickets stay in the trash for 30 days.", "product-faq"),
         ("single sign-on", "Single sign-on is available on the Business plan and above.", "product-faq")]

class UpstreamDown(Exception): pass      # the vector store or the model provider does not answer
class DeadlineExceeded(Exception): pass  # the time allowed for the request is used up

def pipeline(tenant, question, top_k):
    for phrase, text, doc in FACTS:
        if phrase in question.lower():
            return {"answer": text, "citations": [doc]}
    return {"answer": "I could not find that in the documents.", "citations": []}

ERRORS = {  # code: (status line, may the caller retry the same request?)
    "invalid_json": ("400 Bad Request", False),
    "unauthorized": ("401 Unauthorized", False),
    "not_found": ("404 Not Found", False),
    "method_not_allowed": ("405 Method Not Allowed", False),
    "payload_too_large": ("413 Content Too Large", False),
    "unsupported_media_type": ("415 Unsupported Media Type", False),
    "invalid_request": ("422 Unprocessable Content", False),
    "rate_limited": ("429 Too Many Requests", True),
    "internal_error": ("500 Internal Server Error", True),
    "upstream_unavailable": ("503 Service Unavailable", True),
    "deadline_exceeded": ("504 Gateway Timeout", True),
}

class ApiError(Exception):
    def __init__(self, code, message, details=None, headers=()):
        super().__init__(message)
        self.code, self.message, self.details, self.headers = code, message, details, list(headers)

def error_body(error, request_id):
    body = {"code": error.code, "message": error.message, "retryable": ERRORS[error.code][1], "request_id": request_id}
    if error.details:
        body["details"] = error.details
    return {"error": body}

def validate(data):
    """returns (clean request, problems); every problem names its field"""
    if not isinstance(data, dict):
        return None, [{"field": "body", "problem": "must be a JSON object"}]
    problems = []
    def bad(field, problem):
        problems.append({"field": field, "problem": problem})
    question = data.get("question")
    if question is None:
        bad("question", "is required")
    elif not isinstance(question, str):
        bad("question", "must be a string")
    elif not question.strip():
        bad("question", "must not be empty")
    elif len(question) > 500:
        bad("question", "must be at most 500 characters")
    top_k = data.get("top_k", 4)
    if type(top_k) is not int:            # type(), because True would pass isinstance(top_k, int)
        bad("top_k", "must be an integer")
    elif not 1 <= top_k <= 10:
        bad("top_k", "must be from 1 to 10")
    for name in sorted(set(data) - {"question", "top_k"}):
        bad(name, "unknown field")
    if problems:
        return None, problems
    return {"question": question.strip(), "top_k": top_k}, []

from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

class AskRequest(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)   # unknown fields refused; true and "7" are not integers
    question: str
    top_k: int = Field(default=4, ge=1, le=10)

    @field_validator("question")
    @classmethod
    def clean_question(cls, value):
        if not value.strip():
            raise ValueError("must not be empty")
        if len(value) > 500:
            raise ValueError("must be at most 500 characters")
        return value.strip()

WORDS = {"missing": "is required", "string_type": "must be a string", "int_type": "must be an integer",
         "greater_than_equal": "must be from 1 to 10", "less_than_equal": "must be from 1 to 10", "extra_forbidden": "unknown field"}

def problems_of(error):
    """pydantic's error list, translated into the details of our contract"""
    found = []
    for e in error.errors():
        if not e["loc"]:
            found.append({"field": "body", "problem": "must be a JSON object"})
            continue
        field = e["loc"][0]
        if field == "question" and e["input"] is None:
            text = "is required"
        else:
            text = WORDS.get(e["type"]) or str(e["ctx"]["error"])      # the text of our own ValueError
        found.append({"field": field, "problem": text})
    return sorted(found, key=lambda d: ({"question": 0, "top_k": 1}.get(d["field"], 2), d["field"]))

import math
from fastapi import FastAPI, Request
from fastapi.concurrency import run_in_threadpool
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from starlette.exceptions import HTTPException as RoutingError

KEYS = {"demo-key-acme": "acme", "demo-key-zen": "zen"}     # invented demo keys: API key -> tenant
MAX_BYTES = 4096

def make_api(run=pipeline, limiter=None):
    api = FastAPI(redirect_slashes=False, docs_url=None, redoc_url=None, openapi_url=None)
    counter = [0]

    def request_id(request):
        if not hasattr(request.state, "rid"):
            counter[0] += 1
            given = request.headers.get("x-request-id", "")
            request.state.rid = given if re.fullmatch(r"[A-Za-z0-9-]{1,64}", given) else f"req-{counter[0]:04d}"
        return request.state.rid

    def failure(request, code, message, details=None, headers=()):
        error, rid = ApiError(code, message, details), request_id(request)
        return JSONResponse(error_body(error, rid), status_code=int(ERRORS[code][0][:3]),
                            headers={**dict(headers), "X-Request-ID": rid})

    @api.exception_handler(ApiError)
    async def our_error(request, e):
        return failure(request, e.code, e.message, e.details, e.headers)

    @api.exception_handler(RoutingError)                    # the framework's own 404 and 405
    async def routing_error(request, e):
        if e.status_code == 405:
            return failure(request, "method_not_allowed", "use POST", headers=[("Allow", "POST")])
        return failure(request, "not_found", "no such path")

    @api.exception_handler(UpstreamDown)
    async def upstream(request, e):
        return failure(request, "upstream_unavailable", "a dependency is not answering", headers=[("Retry-After", "5")])

    @api.exception_handler(DeadlineExceeded)
    async def too_slow(request, e):
        return failure(request, "deadline_exceeded", "the answer took too long")

    @api.exception_handler(Exception)                       # everything else: say nothing about the inside
    async def bug_in_our_code(request, e):
        return failure(request, "internal_error", "something went wrong on our side")

    @api.post("/v1/ask")
    async def ask(request: Request):
        scheme, _, key = request.headers.get("authorization", "").partition(" ")
        if scheme != "Bearer" or key not in KEYS:
            raise ApiError("unauthorized", "missing or unknown API key", headers=[("WWW-Authenticate", "Bearer")])
        if limiter:
            allowed, wait = limiter(KEYS[key])
            if not allowed:
                raise ApiError("rate_limited", "too many requests", headers=[("Retry-After", str(math.ceil(wait)))])
        if request.headers.get("content-type", "").split(";")[0].strip() != "application/json":
            raise ApiError("unsupported_media_type", "send application/json")
        raw = await request.body()
        if len(raw) > MAX_BYTES:
            raise ApiError("payload_too_large", "the body is larger than 4096 bytes")
        try:
            data = json.loads(raw)
        except ValueError:
            raise ApiError("invalid_json", "the body is not valid JSON")
        try:
            wanted = AskRequest.model_validate(data)
        except ValidationError as error:
            raise ApiError("invalid_request", "the request does not match the schema", problems_of(error))
        result = await run_in_threadpool(run, KEYS[key], wanted.question, wanted.top_k)   # the pipeline blocks: keep the event loop free
        rid = request_id(request)
        return JSONResponse({**result, "request_id": rid}, headers={"X-Request-ID": rid})

    return api

A valid request, a request with three problems, and a comparison of the pydantic route with the hand-written validation.

In [ ]:
client = TestClient(make_api())
KEY = {"Authorization": "Bearer demo-key-acme"}
r = client.post("/v1/ask", json={"question": "How much annual leave do I get?"}, headers=KEY)
print(r.status_code, r.headers["x-request-id"], r.json()["citations"])
r = client.post("/v1/ask", json={"question": "  ", "top_k": 0, "mood": 1}, headers=KEY)
print(r.status_code, r.json()["error"]["code"])
for d in r.json()["error"]["details"]:
    print("   ", d["field"], "-", d["problem"])

BODIES = [{}, [], {"question": None}, {"question": 5}, {"question": ""}, {"question": "x" * 501}, {"question": "a", "top_k": None},
          {"question": "a", "top_k": 4.0}, {"question": "a", "top_k": True}, {"question": "a", "top_k": 11}, {"question": " a ", "top_k": 10},
          {"z": 1, "b": 2}]
same = 0
for body in BODIES:
    try:
        wanted = AskRequest.model_validate(body)
        mine = ({"question": wanted.question, "top_k": wanted.top_k}, [])
    except ValidationError as error:
        mine = (None, problems_of(error))
    same += mine == validate(body)                           # validate: the hand-written function of the contract
print(f"same result as the hand-written validation: {same} of {len(BODIES)} bodies")

200 req-0001 ['leave-policy']
422 invalid_request
    question - must not be empty
    top_k - must be from 1 to 10
    mood - unknown field
same result as the hand-written validation: 12 of 12 bodies


## 3. The table of contract cases, first against the app of section 1, then against the finished one

In [ ]:
def down(tenant, question, top_k):
    raise UpstreamDown()

def slow(tenant, question, top_k):
    raise DeadlineExceeded()

def bug(tenant, question, top_k):        # an ordinary programming error; its message holds things a caller must not see
    raise RuntimeError("cannot open /srv/index/acme.db with password=demo-secret")

FAULTS = {None: pipeline, "down": down, "slow": slow, "bug": bug}

GOOD = {"question": "How much annual leave do I get?"}
ACME = {"Authorization": "Bearer demo-key-acme"}

def case(name, status, code=None, body=GOOD, headers=ACME, method="POST", path="/v1/ask", fault=None, retryable=False, fields=()):
    return dict(name=name, status=status, code=code, body=body, headers=headers, method=method, path=path, fault=fault, retryable=retryable, fields=list(fields))

CASES = [
    case("valid question", 200),
    case("trailing slash", 404, "not_found", path="/v1/ask/"),
    case("GET instead of POST", 405, "method_not_allowed", method="GET", body=None),
    case("unknown API key", 401, "unauthorized", headers={"Authorization": "Bearer nope"}),
    case("plain text body", 415, "unsupported_media_type", headers=dict(ACME, **{"Content-Type": "text/plain"})),
    case("body over 4,096 bytes", 413, "payload_too_large", body={"question": "x" * 5000}),
    case("broken JSON", 400, "invalid_json", body="{oops"),
    case("question missing", 422, "invalid_request", body={}, fields=["question"]),
    case("top_k is true", 422, "invalid_request", body=dict(GOOD, top_k=True), fields=["top_k"]),
    case("tenant in the body", 422, "invalid_request", body=dict(GOOD, tenant="zen"), fields=["tenant"]),
    case("bad key and bad JSON", 401, "unauthorized", body="{oops", headers={}),
    case("vector store down", 503, "upstream_unavailable", fault="down", retryable=True),
    case("answer too slow", 504, "deadline_exceeded", fault="slow", retryable=True),
    case("bug in the pipeline", 500, "internal_error", fault="bug", retryable=True),
]

def check(c, status, headers, body):
    """the list of broken promises; headers has lower-case names"""
    if status != c["status"]:
        return [f"status {status}, expected {c['status']}"]
    broken = []
    if "x-request-id" not in headers:
        broken.append("no X-Request-ID header")
    if c["status"] == 200:
        if set(body) != {"answer", "citations", "request_id"}:
            broken.append("wrong success fields")
        elif not isinstance(body["citations"], list):
            broken.append("citations is not a list")
        return broken
    error = body.get("error", {})
    names = {"code", "message", "retryable", "request_id"} | ({"details"} if c["fields"] else set())
    if set(error) != names:
        return broken + ["wrong error fields"]
    if (error["code"], error["retryable"]) != (c["code"], c["retryable"]):
        broken.append(f"got {error['code']}, retryable {error['retryable']}")
    if error["request_id"] != headers.get("x-request-id"):
        broken.append("request id differs from the header")
    if [d["field"] for d in error.get("details", [])] != c["fields"]:
        broken.append("details name other fields")
    if status == 405 and headers.get("allow") != "POST":
        broken.append("no Allow header")
    if status == 503 and "retry-after" not in headers:
        broken.append("no Retry-After header")
    if "demo-secret" in json.dumps(body):
        broken.append("internal detail in the response")
    return broken

def run_cases(send, show=True):
    """send(case) performs the request and returns (status, headers, body); returns the failed cases"""
    failed = []
    for c in CASES:
        status, headers, body = send(c)
        broken = check(c, status, {k.lower(): v for k, v in headers.items()}, body)
        if broken:
            failed.append((c["name"], broken))
        if show:
            print(f"{c['name']:22} {status}  {'ok' if not broken else 'FAIL ' + broken[0]}")
    print(f"passed {len(CASES) - len(failed)} of {len(CASES)}")
    return failed

def send_to(api):
    def send(c):
        client = TestClient(api(FAULTS[c["fault"]]), raise_server_exceptions=False)   # we want the 500 response, not the traceback
        headers, content = dict(c["headers"]), None
        if c["body"] is not None:
            content = c["body"] if isinstance(c["body"], str) else json.dumps(c["body"])
            headers.setdefault("Content-Type", "application/json")
        r = client.request(c["method"], c["path"], content=content, headers=headers, follow_redirects=False)
        return r.status_code, r.headers, r.json() if r.content else {}
    return send

In [ ]:
print("types only :", end=" ")
failed = run_cases(send_to(lambda run: first), show=False)    # the app of step 1 has no pipeline to break
print("finished app")
run_cases(send_to(make_api))

types only : passed 0 of 14
finished app
valid question         200  ok
trailing slash         404  ok
GET instead of POST    405  ok
unknown API key        401  ok
plain text body        415  ok
body over 4,096 bytes  413  ok
broken JSON            400  ok
question missing       422  ok
top_k is true          422  ok
tenant in the body     422  ok
bad key and bad JSON   401  ok
vector store down      503  ok
answer too slow        504  ok
bug in the pipeline    500  ok
passed 14 of 14


## 4. A token bucket per tenant in front of the work

In [ ]:
class TokenBucket:
    """holds at most `capacity` tokens and gains `rate` tokens per second; a request takes `cost` tokens"""
    def __init__(self, capacity, rate, now=0.0):
        self.capacity, self.rate = capacity, rate
        self.tokens, self.last = float(capacity), now

    def allow(self, now, cost=1):
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= cost:
            self.tokens -= cost
            return True, 0.0
        return False, (cost - self.tokens) / self.rate      # seconds until enough tokens exist

now = [0.0]                                                  # a fake clock, moved by hand
buckets = {}

def limiter(tenant):
    bucket = buckets.setdefault(tenant, TokenBucket(capacity=3, rate=0.5, now=now[0]))
    return bucket.allow(now[0])

client = TestClient(make_api(limiter=limiter))
Q = {"question": "How much annual leave do I get?"}
for at, tenant in [(0, "acme"), (0, "acme"), (0, "acme"), (0, "acme"), (0, "zen"), (1, "acme"), (2, "acme")]:
    now[0] = at
    r = client.post("/v1/ask", json=Q, headers={"Authorization": f"Bearer demo-key-{tenant}"})
    print(f"second {at} {tenant:4} {r.status_code}  Retry-After: {r.headers.get('retry-after', '-')}")

second 0 acme 200  Retry-After: -
second 0 acme 200  Retry-After: -
second 0 acme 200  Retry-After: -
second 0 acme 429  Retry-After: 2
second 0 zen  200  Retry-After: -
second 1 acme 429  Retry-After: 1
second 2 acme 200  Retry-After: -


## Try next

Add a case for a question of exactly 500 characters, then change the limit to 400 and see which cases notice. Never paste a real secret into a notebook cell.